# 01 — Geração dos cenários sintéticos

Cria o CCD, calibra deterministicamente as âncoras, diagnostica redundância, valida recuperação sem ruído e amostra diretamente o casco convexo. A referência não é descoberta por um otimizador.

In [ ]:
from pathlib import Path
import json, os, time, math, gc
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize, minimize_scalar
from scipy.spatial import ConvexHull, Delaunay, cKDTree
from scipy.stats import qmc

def project_root(start=Path.cwd()):
    p=start.resolve()
    for candidate in (p,*p.parents):
        if (candidate/'configs'/'smoke.json').exists(): return candidate
    raise FileNotFoundError('Raiz do projeto não encontrada')

ROOT=project_root(); MODE=os.environ.get('CNBI_MODE','SMOKE').upper()
if MODE=='FULL' and os.environ.get('CNBI_FULL_CONFIRMED')!='YES':
    raise RuntimeError('FULL bloqueado: defina CNBI_FULL_CONFIRMED=YES após autorização explícita.')
CFG=json.loads((ROOT/'configs'/f'{MODE.lower()}.json').read_text(encoding='utf-8'))
for key in ('OMP_NUM_THREADS','MKL_NUM_THREADS','OPENBLAS_NUM_THREADS','NUMEXPR_NUM_THREADS'): os.environ[key]='1'
ALPHA=2**0.75; DELTA_BY_K={2:.10,3:.10,4:.20,5:.50}

OUT=ROOT/'data'/'generated'; REF=ROOT/'data'/'reference_fronts'
if MODE=='SMOKE': OUT=OUT/'smoke'; REF=REF/'smoke'
OUT.mkdir(parents=True,exist_ok=True); REF.mkdir(parents=True,exist_ok=True)

def ccd3():
    factorial=np.array(list(__import__('itertools').product([-1.,1.],repeat=3)))
    axial=np.vstack([np.eye(3)*ALPHA,-np.eye(3)*ALPHA])
    return np.vstack([factorial,axial,np.zeros((5,3))])

def design(X):
    x1,x2,x3=np.asarray(X).T
    return np.column_stack([np.ones(len(x1)),x1,x2,x3,x1*x1,x2*x2,x3*x3,x1*x2,x1*x3,x2*x3])

X_CCD=ccd3(); Z=design(X_CCD)
assert X_CCD.shape==(19,3) and np.linalg.matrix_rank(Z)==10

def uniform_ball_sobol(n,seed):
    eng=qmc.Sobol(3,scramble=True,seed=seed); chunks=[]; total=0
    while total<n:
        u=eng.random_base2(15) if total==0 else eng.random(32768)
        y=2*u-1; y=y[np.einsum('ij,ij->i',y,y)<=1]
        chunks.append(y); total+=len(y)
    return np.vstack(chunks)[:n]*ALPHA

def base_directions(m):
    if m==4:
        u=np.array([[1,1,1],[1,-1,-1],[-1,1,-1],[-1,-1,1]],float)
        return u/np.linalg.norm(u,axis=1,keepdims=True)
    i=np.arange(m); z=1-2*(i+.5)/m; phi=np.pi*(1+np.sqrt(5))*i; r=np.sqrt(1-z*z)
    return np.column_stack([r*np.cos(phi),r*np.sin(phi),z])

def anchors(m,c):
    u=base_directions(m); pole=np.array([1.,.31,-.19]); pole/=np.linalg.norm(pole)
    d=(1-c)*u+c*pole; d/=np.linalg.norm(d,axis=1,keepdims=True)
    A=.65*ALPHA*d
    assert np.linalg.matrix_rank(A[1:]-A[0],tol=1e-10)==3
    return A

def truth(X,A): return np.sum((np.asarray(X)[:,None,:]-A[None,:,:])**2,axis=2)
def rho_abs(F):
    C=np.corrcoef(F,rowvar=False); return float(np.mean(np.abs(C[np.triu_indices(C.shape[0],1)]))),C

X_CAL=uniform_ball_sobol(int(CFG['sobol_calibration_points']),2026)
rows=[]
for m in CFG['scenario_objectives']:
  for level,target in CFG['correlation_targets'].items():
    def loss(c): return abs(rho_abs(truth(X_CAL,anchors(m,float(c))))[0]-target)
    grid=np.linspace(0,.97,195); vals=np.array([loss(c) for c in grid]); c0=grid[vals.argmin()]
    lo=max(0,c0-.02); hi=min(.98,c0+.02); opt=minimize_scalar(loss,bounds=(lo,hi),method='bounded',options={'xatol':1e-10})
    A=anchors(m,float(opt.x)); F=truth(X_CAL,A); rho,C=rho_abs(F)
    s=np.linalg.svd((F-F.mean(0))/F.std(0,ddof=1),compute_uv=False); lam=s*s; p=lam/lam.sum(); reff=float(np.exp(-np.sum(p[p>0]*np.log(p[p>0]))))
    sid=f'm{m}_{level}'; np.savez_compressed(OUT/f'{sid}_scenario.npz',anchors=A,correlation=C,singular_values=s)
    rows.append({'scenario':sid,'m':m,'level':level,'target':target,'achieved':rho,'deviation':rho-target,'concentration':float(opt.x),'affine_rank':3,'effective_rank':reff,'redundancy':1-reff/m,'within_tolerance':abs(rho-target)<=CFG['correlation_tolerance']})
scenarios=pd.DataFrame(rows); scenarios.to_csv(OUT/'scenario_diagnostics.csv',index=False)
assert len(scenarios)==len(CFG['scenario_objectives'])*len(CFG['correlation_targets'])

def fit_rsm(A,seed,noisy=True):
    F=truth(X_CCD,A); rng=np.random.default_rng(seed); var=F.var(0,ddof=1); sigma=np.sqrt(var*(1-.95)/.95)
    Y=F+rng.normal(0,sigma,size=F.shape) if noisy else F
    B=np.linalg.lstsq(Z,Y,rcond=None)[0]; Yh=Z@B; E=Y-Yh; mse=np.sum(E*E,axis=0)/(len(Y)-Z.shape[1]); r2=1-np.sum(E*E,axis=0)/np.sum((Y-Y.mean(0))**2,axis=0)
    return B,mse,r2,sigma

for rec in rows:
    A=np.load(OUT/f"{rec['scenario']}_scenario.npz")['anchors']; B0,_,_,_=fit_rsm(A,101,False)
    probe=uniform_ball_sobol(1024,91); err=np.max(np.abs(design(probe)@B0-truth(probe,A)))
    assert err<1e-10

def sample_convex_hull(A,n,seed):
    tri=Delaunay(A); T=A[tri.simplices]; vols=np.abs(np.linalg.det(T[:,1:]-T[:,:1]))/6
    rng=np.random.default_rng(seed); ids=rng.choice(len(T),size=n,p=vols/vols.sum())
    e=rng.exponential(size=(n,4)); w=e/e.sum(1,keepdims=True); X=np.einsum('ni,nij->nj',w,T[ids])
    assert np.max(np.linalg.norm(X,axis=1))<=ALPHA+1e-10
    return X

for rec in rows:
    A=np.load(OUT/f"{rec['scenario']}_scenario.npz")['anchors']; Xp=sample_convex_hull(A,int(CFG['reference_points']),404)
    np.savez_compressed(REF/f"{rec['scenario']}_pareto_reference.npz",X=Xp,F=truth(Xp,A))
print(scenarios[['scenario','target','achieved','within_tolerance','effective_rank']].to_string(index=False))


## Auditoria numérica dos cenários

Verifica explicitamente âncoras, ótimos, espectro, reprodutibilidade, casco e não dominância aproximada da referência.

In [ ]:
# EXPLICIT SCENARIO NUMERICAL AUDIT
audit=[]
for rec in rows:
    sid=rec['scenario']; A=np.load(OUT/f'{sid}_scenario.npz')['anchors']; m=len(A)
    pairwise=np.linalg.norm(A[:,None,:]-A[None,:,:],axis=2); pairwise[np.diag_indices(m)]=np.inf
    assert pairwise.min()>1e-8 and np.max(np.linalg.norm(A,axis=1))<ALPHA and np.linalg.matrix_rank(A[1:]-A[0])==3
    anchor_values=truth(A,A); assert np.max(np.abs(np.diag(anchor_values)))<1e-12
    probe1=sample_convex_hull(A,256,5150+m); probe2=sample_convex_hull(A,256,5150+m); assert np.array_equal(probe1,probe2)
    assert np.all(Delaunay(A).find_simplex(probe1)>=0)
    ref=np.load(REF/f'{sid}_pareto_reference.npz'); Fref=ref['F']; rng=np.random.default_rng(8080+m); ids=rng.choice(len(Fref),size=min(512,len(Fref)),replace=False); Fs=Fref[ids]
    dominated=np.array([np.any(np.all(Fs<=f+1e-12,axis=1)&np.any(Fs<f-1e-10,axis=1)) for f in Fs]); assert dominated.mean()<.01
    Fcal=truth(X_CAL,A); standardized=(Fcal-Fcal.mean(0))/Fcal.std(0,ddof=1); singular=np.linalg.svd(standardized,compute_uv=False); variance=singular**2; cumulative=np.cumsum(variance)/variance.sum(); numerical_rank=int(np.linalg.matrix_rank(standardized,tol=1e-10))
    audit.append({'scenario':sid,'min_anchor_distance':float(pairwise.min()),'max_anchor_norm':float(np.linalg.norm(A,axis=1).max()),'numerical_rank':numerical_rank,'singular_values_json':json.dumps(singular.tolist()),'cumulative_variance_json':json.dumps(cumulative.tolist()),'reference_reproducible':True,'reference_inside_hull':True,'reference_approx_nondominated_fraction':float(1-dominated.mean()),'known_optima_verified':True})
audit=pd.DataFrame(audit); scenarios=scenarios.drop(columns=[c for c in audit.columns if c!='scenario' and c in scenarios],errors='ignore').merge(audit,on='scenario',validate='one_to_one'); scenarios.to_csv(OUT/'scenario_diagnostics.csv',index=False)
for m,g in scenarios.groupby('m'):
    ordered=g.set_index('level').loc[[x for x in ('low','medium','high') if x in set(g.level)]]
    if len(ordered)>1: assert np.all(np.diff(ordered.achieved)>=-1e-8),f'Correlação não monotônica em m={m}'
print(f'Auditoria explícita aprovada para {len(audit)} cenários.')